# PARTE 4 — Materialized Views: control incremental automático

> ⚠️ **Este notebook debe ejecutarse conectado a un SQL Warehouse**, no a un cluster Spark estándar.
> En Databricks, selecciona el SQL Warehouse en el desplegable de compute antes de ejecutar cualquier celda.

---

## ¿Por qué Materialized Views?

En el notebook anterior hemos visto Structured Streaming: tú defines el `readStream`, las transformaciones, el `writeStream`, el trigger, el checkpoint. Tienes control total — y toda la responsabilidad.

Pero hay una pregunta legítima: **¿y si no necesito todo ese control?**

Ya tienes Bronze cargado gracias al stream de Auto Loader. Ahora quieres construir una capa Silver que:
- Filtre lecturas anómalas
- Clasifique la temperatura de cada lectura
- Enriquezca los datos con el estado de la máquina en ese momento
- Se actualice automáticamente cuando Bronze recibe datos nuevos

Con Structured Streaming para Silver necesitarías un stream-stream join con watermarks, gestión del State Store, output mode restringido... la complejidad es alta.

**Las Materialized Views resuelven esto con SQL puro.** Tú defines el resultado que quieres. Databricks decide cómo mantenerlo actualizado de forma incremental.

## ¿Qué es Enzyme?

Enzyme es el motor interno de Databricks que hace que las Materialized Views sean incrementales. Cuando defines una MV, Enzyme:

1. Analiza el query plan usando Catalyst (el optimizador de Spark)
2. Lee el `_delta_log` de la tabla fuente para saber exactamente qué ha cambiado desde el último refresh
3. Elige la estrategia más barata para aplicar esos cambios:

| Estrategia | Cuándo la usa |
|---|---|
| **Monotonic Append** | Solo han llegado filas nuevas — la más barata |
| **Partition Recompute** | Han cambiado datos en particiones específicas |
| **MERGE Updates** | Ha habido updates o deletes |
| **Full Recompute** | El query es demasiado complejo para incrementalizar |

Tú no ves nada de esto — Enzyme lo decide solo.

## Por qué las MVs solo funcionan en SQL Warehouse

Las Materialized Views son una feature del motor SQL de Databricks (Photon), no de Apache Spark. Necesitan acceso profundo a Unity Catalog para gestionar el ciclo de vida de los objetos, detectar cambios en las tablas fuente y coordinar los refreshes. Esa integración solo existe en el SQL Warehouse, no en el cluster Spark estándar.

| | Cluster Spark (notebook estándar) | SQL Warehouse |
|---|---|---|
| PySpark / streaming | ✅ | ❌ |
| SQL general | ✅ | ✅ |
| Materialized Views | ❌ | ✅ |
| Streaming Tables | ❌ | ✅ |

## Diferencia fundamental con Structured Streaming

| | Structured Streaming | Materialized View |
|---|---|---|
| Modelo | Microbatch continuo o programado | Batch incremental programado |
| Quién gestiona el estado | Tú (checkpoint, watermarks) | Enzyme automáticamente |
| Fuente | S3, Kafka, Delta... | Solo tablas Delta |
| Joins complejos | Stream-stream join con watermarks | SQL JOIN estándar |
| Operaciones no soportadas | ORDER BY, DISTINCT global... | SQL completo |
| Código necesario | readStream + transform + writeStream | Solo SQL |

<img src="https://raw.githubusercontent.com/jmartinezceste/Course_Delta_Lake/main/delta_img/delta_ses3_3.png" width="800px"/>

## 4.1 Setup: schemas Bronze y Silver

Creamos dos schemas separados para mantener la arquitectura medallion limpia:
- `sesion_mv_br` → Bronze: datos crudos ingestados por el stream de Auto Loader (notebook anterior)
- `sesion_mv_sv` → Silver: datos limpios y enriquecidos por Materialized Views

Los datos de Bronze ya existen — los cargó el stream de Auto Loader. Aquí solo creamos el schema Silver y verificamos que Bronze tiene datos.

In [0]:
-- ── CELL 1: Setup schemas ────────────────────────────────────────
CREATE SCHEMA IF NOT EXISTS sesion_mv_br;
CREATE SCHEMA IF NOT EXISTS sesion_mv_sv;

In [0]:
-- ── CELL 2: Verificar Bronze ─────────────────────────────────────
-- Comprobamos que las tablas Bronze existen y tienen datos
-- Si están vacías, vuelve al notebook de streaming y ejecuta los streams primero
SELECT 'sensor_readings' AS tabla, COUNT(*) AS num_registros FROM sesion_mv_br.sensor_readings
UNION ALL
SELECT 'machine_events',           COUNT(*)                  FROM sesion_mv_br.machine_events;

tabla,num_registros
sensor_readings,40
machine_events,15


In [0]:
-- ── CELL 3: Preview Bronze sensor_readings ───────────────────────
SELECT * FROM sesion_mv_br.sensor_readings
ORDER BY timestamp DESC
LIMIT 10;

id,id_machine,temperature,timestamp,_rescued_data,ingested_at
210ca6d2-e602-4333-8645-1ae200f7d55d,machine_3,27.03,2026-05-07T11:45:29.506822,null,2026-05-07T11:45:59.510Z
bfda5a02-a11f-4432-95ba-b3ceba205217,machine_2,32.64,2026-05-07T11:45:26.579228,null,2026-05-07T11:45:59.510Z
4e27694c-864d-44f9-9616-1403c0e05d01,machine_2,24.47,2026-05-07T11:45:23.517918,null,2026-05-07T11:45:59.510Z
342ef09e-1b5a-4833-83a6-7e7b966b1140,machine_3,32.18,2026-05-07T11:45:20.647623,null,2026-05-07T11:45:59.510Z
bc67ade0-db7d-4cb4-87c3-2b32130d7a05,machine_5,22.25,2026-05-07T11:45:17.749228,null,2026-05-07T11:45:59.510Z
e44fc615-6058-4fba-a000-7f567f57e79a,machine_5,20.55,2026-05-07T11:45:14.885461,null,2026-05-07T11:45:59.510Z
c9102b18-932e-41fc-9cf0-2a11fbbad6fc,machine_2,24.2,2026-05-07T11:45:11.854296,null,2026-05-07T11:45:59.510Z
27287d85-502e-4e50-87ac-60aa5e8168a3,machine_1,22.69,2026-05-07T11:45:08.984491,null,2026-05-07T11:45:59.510Z
160db000-d87f-4dfa-b273-b94845a25a35,machine_1,17.92,2026-05-07T11:45:06.086661,null,2026-05-07T11:45:59.510Z
8d17cc1b-79fd-4960-8375-6afeeef43268,machine_5,29.05,2026-05-07T11:45:02.980433,null,2026-05-07T11:45:59.510Z


In [0]:
-- ── CELL 4: Preview Bronze machine_events ────────────────────────
SELECT * FROM sesion_mv_br.machine_events
ORDER BY timestamp DESC
LIMIT 10;

id,id_machine,status,timestamp,_rescued_data,ingested_at
6abec4b1-7329-43ca-b15a-2843f7af1b69,machine_5,maintenance,2026-05-07T11:45:40.468486,null,2026-05-07T11:46:31.073Z
a0efab18-ba2d-40b3-8636-44a3814af576,machine_4,error,2026-05-07T11:45:38.594419,null,2026-05-07T11:46:31.073Z
be37c7a6-678c-405c-a7ba-f8836febf33c,machine_3,error,2026-05-07T11:45:36.676345,null,2026-05-07T11:46:31.073Z
ddc12444-0409-4a1e-8aaf-ccb2698e9bc8,machine_2,online,2026-05-07T11:45:34.637731,null,2026-05-07T11:46:31.073Z
ec8ecdae-4f30-4dc4-9c51-87b540990935,machine_1,offline,2026-05-07T11:45:32.585611,null,2026-05-07T11:46:31.073Z
da77a677-90a4-4c53-aa9c-3d42ef094b50,machine_5,maintenance,2026-05-07T09:28:54.164608,null,2026-05-07T09:29:53.082Z
ff780ca1-68df-445e-9ac1-ddcf9b115b44,machine_4,error,2026-05-07T09:28:52.197213,null,2026-05-07T09:29:53.082Z
2b4f0bff-f38f-4fb2-b6b9-586c3b749a8e,machine_3,online,2026-05-07T09:28:50.279676,null,2026-05-07T09:29:53.082Z
1d7ca990-5da6-4e40-8840-f50cb5668dbc,machine_2,online,2026-05-07T09:28:48.201104,null,2026-05-07T09:29:53.082Z
6c9cafba-e181-43c1-ac71-eefe14c6a68f,machine_1,offline,2026-05-07T09:28:46.072575,null,2026-05-07T09:29:53.082Z


## 4.2 MV 1 — Lecturas válidas con flag de anomalía

La primera Materialized View aplica dos cosas sobre Bronze:

**Filtrado**: descarta lecturas con temperatura `NULL` o físicamente imposibles (fuera del rango 0–100°C). En Bronze guardamos los datos tal como llegaron — sin transformar ni filtrar. Bronze es inmutable. Silver es donde aplicamos las reglas de calidad.

**Clasificación**: añade `anomaly_flag` que categoriza cada lectura:
- `high` → temperatura superior a 28°C
- `low` → temperatura inferior a 15°C  
- `normal` → dentro del rango esperado

Esta columna calculada en Structured Streaming sería un `.withColumn()` con `when/otherwise`. Aquí es un `CASE WHEN` en SQL. El resultado es idéntico.

**Lo que Enzyme hará internamente**: como Bronze solo recibe inserciones (append), Enzyme detectará que puede usar la estrategia **Monotonic Append** — la más barata. Solo procesará las filas nuevas de Bronze, sin tocar las ya procesadas.

In [0]:
-- ── CELL 5: MV 1 — Lecturas válidas con anomaly_flag ─────────────
CREATE OR REPLACE MATERIALIZED VIEW sesion_mv_sv.valid_readings AS
SELECT
    id,
    id_machine,
    temperature,
    timestamp,
    ingested_at,
    CASE
        WHEN temperature > 28.0 THEN 'high'
        WHEN temperature < 15.0 THEN 'low'
        ELSE 'normal'
    END AS anomaly_flag
FROM sesion_mv_br.sensor_readings
WHERE temperature IS NOT NULL
  AND temperature BETWEEN 0.0 AND 100.0;  -- filtro de valores físicamente imposibles

result
The operation was successfully executed.


In [0]:
-- ── CELL 6: Verificar MV 1 ───────────────────────────────────────
SELECT
    anomaly_flag,
    COUNT(*)                     AS num_lecturas,
    ROUND(AVG(temperature), 2)   AS avg_temperature,
    ROUND(MIN(temperature), 2)   AS min_temperature,
    ROUND(MAX(temperature), 2)   AS max_temperature
FROM sesion_mv_sv.valid_readings
GROUP BY anomaly_flag
ORDER BY anomaly_flag;

anomaly_flag,num_lecturas,avg_temperature,min_temperature,max_temperature
high,11,32.02,29.05,34.84
normal,29,22.05,16.23,27.89


## 4.3 MV 2 — Estado más reciente por máquina

La segunda Materialized View responde una pregunta de negocio importante: **¿cuál es el estado actual de cada máquina?**

Bronze guarda todos los eventos de estado históricos — cada vez que una máquina cambia de estado llega un nuevo registro. En Silver queremos solo el estado más reciente de cada máquina.

Usamos `ROW_NUMBER() OVER (PARTITION BY id_machine ORDER BY timestamp DESC)` con `QUALIFY` para quedarnos con el registro más reciente por máquina.

**Por qué esto es complejo en Structured Streaming:**
Esta es una **agregación con estado** — el resultado de una máquina puede cambiar en cada batch cuando llega un nuevo evento. Necesitarías `outputMode("update")` y Spark mantendría el estado de todas las máquinas en memoria (State Store). Con watermarks tendrías que decidir cuánto tiempo esperar antes de descartar el estado de una máquina que ya no emite eventos.

Con una Materialized View es SQL estándar. Enzyme gestiona el estado automáticamente.

In [0]:
-- ── CELL 7: MV 2 — Estado más reciente por máquina ───────────────
CREATE OR REPLACE MATERIALIZED VIEW sesion_mv_sv.latest_machine_status AS
SELECT
    id_machine,
    status,
    timestamp AS status_timestamp
FROM sesion_mv_br.machine_events
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id_machine
    ORDER BY timestamp DESC
) = 1;

result
The operation was successfully executed.


In [0]:
-- ── CELL 8: Verificar MV 2 ───────────────────────────────────────
-- Debe mostrar exactamente un registro por máquina con su estado más reciente
SELECT *
FROM sesion_mv_sv.latest_machine_status
ORDER BY id_machine;

id_machine,status,status_timestamp
machine_1,offline,2026-05-07T11:45:32.585611
machine_2,online,2026-05-07T11:45:34.637731
machine_3,error,2026-05-07T11:45:36.676345
machine_4,error,2026-05-07T11:45:38.594419
machine_5,maintenance,2026-05-07T11:45:40.468486


## 4.4 El momento clave: el refresh automático

Ahora vamos a demostrar el contraste directo con Structured Streaming.

**Flujo:**
1. Generamos nuevos JSONs en S3 desde el notebook de streaming
2. Relanzamos los streams de Bronze en el notebook de streaming (Cells 20 y 21)
3. Ejecutamos `REFRESH MATERIALIZED VIEW` aquí
4. Las MVs Silver se actualizan con los datos nuevos

**El contraste con Streaming:**
En el notebook de streaming, para procesar datos nuevos en Silver necesitarías definir un nuevo writeStream, gestionar su checkpoint, y si hay un join, implementar un stream-stream join complejo. Aquí el único paso adicional es el `REFRESH`.

En producción ni siquiera ejecutarías el `REFRESH` manualmente — lo haría un **Databricks Job** programado que arranca automáticamente después de que el stream de Bronze termine.

In [0]:
-- ── CELL 9: Refresh MV 1 ─────────────────────────────────────────
-- Ejecuta esto DESPUÉS de haber relanzado los streams de Bronze
-- Enzyme detectará las filas nuevas en Bronze y solo procesará esas
REFRESH MATERIALIZED VIEW sesion_mv_sv.valid_readings;

result
The operation was successfully executed.


In [0]:
-- ── CELL 10: Refresh MV 2 ────────────────────────────────────────
REFRESH MATERIALIZED VIEW sesion_mv_sv.latest_machine_status;

result
The operation was successfully executed.


In [0]:
-- ── CELL 11: Verificar que las MVs tienen datos nuevos ───────────
-- Compara el COUNT antes y después del refresh
SELECT 'valid_readings'        AS mv, COUNT(*) AS num_registros FROM sesion_mv_sv.valid_readings
UNION ALL
SELECT 'latest_machine_status', COUNT(*)                        FROM sesion_mv_sv.latest_machine_status;

mv,num_registros
valid_readings,40
latest_machine_status,5


## 4.5 MV 3 — Join Silver: lecturas enriquecidas con estado de máquina

La Materialized View final cruza las dos anteriores. El resultado responde la pregunta más valiosa para operaciones:

**¿Qué temperatura estaba marcando cada máquina, y en qué estado estaba cuando lo hizo?**

Una lectura `anomaly_flag = 'high'` en una máquina con `machine_status = 'error'` tiene un significado operacional completamente distinto a la misma lectura en una máquina con `machine_status = 'online'`.

### Por qué este join es complejo en Structured Streaming

Para hacer este join con Structured Streaming necesitarías un **stream-stream join** con:
- Watermarks en ambos lados para controlar el estado en memoria (State Store)
- Una time range condition obligatoria
- `outputMode("append")` — el único compatible con stream-stream joins
- Gestión del SymmetricHashJoin — el join simétrico donde cada registro que llega busca match en el store del otro lado y se guarda en el suyo

Con una Materialized View es un `LEFT JOIN` estándar. Enzyme resuelve la incrementalidad.

In [0]:
-- ── CELL 12: MV 3 — Lecturas enriquecidas con estado de máquina ──
CREATE OR REPLACE MATERIALIZED VIEW sesion_mv_sv.enriched_readings AS
SELECT
    r.id                AS reading_id,
    r.id_machine,
    r.temperature,
    r.anomaly_flag,
    r.timestamp         AS reading_timestamp,
    m.status            AS machine_status,
    m.status_timestamp,
    r.ingested_at
FROM sesion_mv_sv.valid_readings r
LEFT JOIN sesion_mv_sv.latest_machine_status m
    ON r.id_machine = m.id_machine;

result
The operation was successfully executed.


In [0]:
-- ── CELL 13: Refresh MV 3 y resultado final ──────────────────────
REFRESH MATERIALIZED VIEW sesion_mv_sv.enriched_readings;

result
The operation was successfully executed.


In [0]:
-- ── CELL 14: Ver resultado final ─────────────────────────────────
SELECT *
FROM sesion_mv_sv.enriched_readings
ORDER BY reading_timestamp DESC
LIMIT 20;

reading_id,id_machine,temperature,anomaly_flag,reading_timestamp,machine_status,status_timestamp,ingested_at
210ca6d2-e602-4333-8645-1ae200f7d55d,machine_3,27.03,normal,2026-05-07T11:45:29.506822,error,2026-05-07T11:45:36.676345,2026-05-07T11:45:59.510Z
bfda5a02-a11f-4432-95ba-b3ceba205217,machine_2,32.64,high,2026-05-07T11:45:26.579228,online,2026-05-07T11:45:34.637731,2026-05-07T11:45:59.510Z
4e27694c-864d-44f9-9616-1403c0e05d01,machine_2,24.47,normal,2026-05-07T11:45:23.517918,online,2026-05-07T11:45:34.637731,2026-05-07T11:45:59.510Z
342ef09e-1b5a-4833-83a6-7e7b966b1140,machine_3,32.18,high,2026-05-07T11:45:20.647623,error,2026-05-07T11:45:36.676345,2026-05-07T11:45:59.510Z
bc67ade0-db7d-4cb4-87c3-2b32130d7a05,machine_5,22.25,normal,2026-05-07T11:45:17.749228,maintenance,2026-05-07T11:45:40.468486,2026-05-07T11:45:59.510Z
e44fc615-6058-4fba-a000-7f567f57e79a,machine_5,20.55,normal,2026-05-07T11:45:14.885461,maintenance,2026-05-07T11:45:40.468486,2026-05-07T11:45:59.510Z
c9102b18-932e-41fc-9cf0-2a11fbbad6fc,machine_2,24.2,normal,2026-05-07T11:45:11.854296,online,2026-05-07T11:45:34.637731,2026-05-07T11:45:59.510Z
27287d85-502e-4e50-87ac-60aa5e8168a3,machine_1,22.69,normal,2026-05-07T11:45:08.984491,offline,2026-05-07T11:45:32.585611,2026-05-07T11:45:59.510Z
160db000-d87f-4dfa-b273-b94845a25a35,machine_1,17.92,normal,2026-05-07T11:45:06.086661,offline,2026-05-07T11:45:32.585611,2026-05-07T11:45:59.510Z
8d17cc1b-79fd-4960-8375-6afeeef43268,machine_5,29.05,high,2026-05-07T11:45:02.980433,maintenance,2026-05-07T11:45:40.468486,2026-05-07T11:45:59.510Z


In [0]:
-- ── CELL 15: Análisis — anomalías por máquina y estado ───────────
-- La pregunta de negocio más relevante:
-- ¿hay lecturas de temperatura alta en máquinas que están en error o mantenimiento?
SELECT
    id_machine,
    machine_status,
    anomaly_flag,
    COUNT(*)                     AS num_lecturas,
    ROUND(AVG(temperature), 2)   AS avg_temperature,
    ROUND(MAX(temperature), 2)   AS max_temperature
FROM sesion_mv_sv.enriched_readings
GROUP BY id_machine, machine_status, anomaly_flag
ORDER BY id_machine, machine_status;

id_machine,machine_status,anomaly_flag,num_lecturas,avg_temperature,max_temperature
machine_1,offline,normal,5,21.62,27.05
machine_1,offline,high,1,33.01,33.01
machine_2,online,normal,4,22.07,24.47
machine_2,online,high,3,32.15,32.64
machine_3,error,normal,6,23.02,27.89
machine_3,error,high,4,31.92,34.84
machine_4,error,normal,5,22.54,26.6
machine_4,error,high,1,34.36,34.36
machine_5,maintenance,normal,9,21.34,26.3
machine_5,maintenance,high,2,30.4,31.74


In [0]:
-- ── CELL 16: Caso crítico — lecturas anómalas en máquinas con error
-- Esto es lo que en producción dispararía una alerta
SELECT
    id_machine,
    machine_status,
    temperature,
    anomaly_flag,
    reading_timestamp
FROM sesion_mv_sv.enriched_readings
WHERE anomaly_flag != 'normal'
  AND machine_status IN ('error', 'maintenance')
ORDER BY reading_timestamp DESC;

id_machine,machine_status,temperature,anomaly_flag,reading_timestamp
machine_3,error,32.18,high,2026-05-07T11:45:20.647623
machine_5,maintenance,29.05,high,2026-05-07T11:45:02.980433
machine_3,error,29.81,high,2026-05-07T09:28:33.607292
machine_3,error,34.84,high,2026-05-07T09:28:30.650368
machine_5,maintenance,31.74,high,2026-05-07T09:28:21.632463
machine_4,error,34.36,high,2026-05-06T14:41:02.454063
machine_3,error,30.84,high,2026-05-06T14:35:58.602667


## 4.6 Linaje de las Materialized Views

Una de las ventajas de las MVs en Databricks es que Unity Catalog registra automáticamente el **linaje** — qué tablas alimentan a qué MVs y en qué orden.

Puedes verlo en la UI de Unity Catalog → Data Explorer → selecciona la MV → pestaña Lineage.

El grafo de dependencias de este pipeline es:

```
sesion_mv_br.sensor_readings  ──▶  sesion_mv_sv.valid_readings         ──▶  sesion_mv_sv.enriched_readings
sesion_mv_br.machine_events   ──▶  sesion_mv_sv.latest_machine_status  ──▶  sesion_mv_sv.enriched_readings
```

Este linaje es automático — no lo has declarado en ningún sitio. Unity Catalog lo infiere del SQL de cada MV.

In [0]:
-- ── CELL 17: Describir la MV enriquecida ─────────────────────────
-- Muestra metadatos, schema y propiedades de la MV
DESCRIBE EXTENDED sesion_mv_sv.enriched_readings;

col_name,data_type,comment
reading_id,string,null
id_machine,string,null
temperature,string,null
anomaly_flag,string,null
reading_timestamp,string,null
machine_status,string,null
status_timestamp,string,null
ingested_at,timestamp,null
,,
# Metadata Columns,,


## Conclusión: Structured Streaming vs Materialized Views

Hemos visto los dos enfoques para procesamiento incremental. El resumen práctico por capa:

| Capa | Herramienta | Motivo |
|---|---|---|
| **Bronze** | Structured Streaming (Auto Loader) | Ingesta desde S3/Kafka directamente. Workload stateless, máxima simplicidad. Las MVs no pueden leer de S3 directamente — necesitan una tabla Delta como fuente |
| **Silver** | Materialized Views | Transformaciones sobre Bronze. Joins, filtros, agregaciones — SQL es suficiente y evita toda la complejidad stateful del streaming |
| **Gold** | Materialized Views o Batch | Agregaciones last-mile. La latencia de minutos/horas es aceptable. SQL es suficiente |

**La regla general**: usa el mínimo nivel de complejidad que resuelve tu problema.

Structured Streaming es poderoso pero tiene un coste de mantenimiento real — checkpoint, watermarks, State Store, output modes. Las Materialized Views delegan ese coste en Databricks y te permiten centrarte en la lógica de negocio.